# BiLSTM

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
# !pip install -q wandb

In [ ]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# wandb_key = user_secrets.get_secret("WANDB_API_KEY")

In [ ]:
# import wandb
# wandb.login(key=wandb_key)

In [ ]:
train=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

OPTIONS=["A", "B", "C", "D", "E"]
MAX_LEN=60
MAX_VOCAB=30000



In [ ]:
print(df.duplicated().sum())

In [ ]:
# Drop duplicates
train =train.drop_duplicates(
    subset=["prompt","A","B","C","D","E"]
)

In [ ]:
tr, val = train_test_split(train, test_size=0.2, random_state=42)
tr.shape, val.shape

In [ ]:
#Converting text to lowercase

def tokenize(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

In [ ]:
# Id Assigning 
def build_vocab(texts, max_vocab=MAX_VOCAB):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    vocab = {"<pad>": 0, "<unk>": 1}
    for word, _ in counter.most_common(max_vocab - len(vocab)):
        vocab[word] = len(vocab)
    return vocab

corpus = pd.concat([train["prompt"], test["prompt"], *[train[o] for o in OPTIONS], *[test[o] for o in OPTIONS]]).astype(str)
vocab = build_vocab(corpus)
len(vocab)

In [ ]:
def encode(text, max_len=MAX_LEN):
    ids = [vocab.get(w, 1) for w in tokenize(text)[:max_len]]
    return ids + [0] * (max_len - len(ids))

class MCQDataset(Dataset):
    def __init__(self, df, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids = torch.tensor(encode(row["prompt"]))
        opt_ids = torch.stack([torch.tensor(encode(row[o])) for o in OPTIONS])
        label = OPTIONS.index(row["answer"]) if self.has_labels else -1
        return prompt_ids, opt_ids, label

In [ ]:
#Creating batches
def collate(batch):
    p = torch.stack([b[0] for b in batch])
    o = torch.stack([b[1] for b in batch])
    y = torch.tensor([b[2] for b in batch])
    return p, o, y

In [ ]:

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden=128, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(hidden * 2, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        mask = (x != 0).float().unsqueeze(-1)
        e = self.dropout(self.emb(x))
        out, _ = self.lstm(e)
        scores = self.attn(out).masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        return (out * weights).sum(1)


class PairScorer(nn.Module):
    def __init__(self, dim, hidden=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 4, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, p, o):
        feat = torch.cat([p, o, torch.abs(p - o), p * o], dim=-1)
        return self.net(feat).squeeze(-1)


class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden=128):
        super().__init__()
        self.encoder = Encoder(vocab_size, emb_dim, hidden)
        self.scorer = PairScorer(hidden * 2)

    def forward(self, prompt_ids, opt_ids):
        B, K, L = opt_ids.shape
        p_vec = self.encoder(prompt_ids)
        o_vec = self.encoder(opt_ids.view(B * K, L)).view(B, K, -1)
        p_vec = p_vec.unsqueeze(1).expand(-1, K, -1)
        return self.scorer(p_vec, o_vec)

In [ ]:
def map_at_3(y_true, y_preds):
    scores = []
    for true, preds in zip(y_true, y_preds):
        score = 0
        for i, pred in enumerate(preds[:3]):
            if pred == true:
                score = 1 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

def rank_from_logits(logits):
    ranked = torch.argsort(logits, dim=1, descending=True)[:, :3].cpu().tolist()
    return [[OPTIONS[i] for i in r] for r in ranked]

def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for p_ids, o_ids, y in loader:
            logits = model(p_ids.to(device), o_ids.to(device))
            preds.extend(rank_from_logits(logits))
            labels.extend(y.tolist())
    y_true = [OPTIONS[l] for l in labels]
    top1 = [p[0] for p in preds]
    acc = accuracy_score(y_true, top1)
    f1 = f1_score(y_true, top1, average="macro")
    map3 = map_at_3(y_true, preds)
    return acc, f1, map3

In [ ]:
model = BiLSTMAttention(len(vocab)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_loader = DataLoader(MCQDataset(tr), batch_size=32, shuffle=True, collate_fn=collate)
val_loader = DataLoader(MCQDataset(val), batch_size=32, shuffle=False, collate_fn=collate)

EPOCHS = 10

In [ ]:
# wandb.init(
#     project="DL-GenAI-2026-t2",
#     entity="23f3002289-dl-genai-project",
#     name="bilstm-attention"
# )

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for p_ids, o_ids, y in train_loader:
        p_ids, o_ids, y = p_ids.to(device), o_ids.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(p_ids, o_ids)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)

    train_loss = total_loss / len(tr)
    acc, f1, map3 = evaluate(model, val_loader)
    print(f"epoch {epoch+1}/{EPOCHS}  loss={train_loss:.4f}  acc={acc:.4f}  f1={f1:.4f}  map@3={map3:.4f}")

    # wandb.log({
    #     "epoch": epoch + 1,
    #     "train_loss": train_loss,
    #     "accuracy": acc,
    #     "f1_macro": f1,
    #     "map_at_3": map3
    # })

In [ ]:
test_loader = DataLoader(MCQDataset(test, has_labels=False), batch_size=64, shuffle=False, collate_fn=collate)

model.eval()
preds = []
with torch.no_grad():
    for p_ids, o_ids, _ in test_loader:
        logits = model(p_ids.to(device), o_ids.to(device))
        preds.extend(rank_from_logits(logits))

submission = pd.DataFrame({"id": test["id"], "Prediction": [" ".join(p) for p in preds]})
submission.to_csv("submission_bilstm_attention.csv", index=False)
submission.head()

In [ ]:
# wandb.finish()